In [7]:
import os
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier

In [8]:
TRAIN_PATH = "/Users/mitulshah/Downloads/mlops-churn-project/data/processed/train.csv"
TEST_PATH = "/Users/mitulshah/Downloads/mlops-churn-project/data/processed/test.csv"


In [9]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

In [10]:
X_train = train_df.drop(columns=["churn"])
y_train = train_df["churn"]

X_test = test_df.drop(columns=["churn"])
y_test = test_df["churn"]

In [11]:
NUMERICAL_FEATURES = [
    "age",
    "tenure_months",
    "monthly_charges",
    "customer_service_calls"
]

CATEGORICAL_FEATURES = [
    "contract_type",
    "internet_service",
    "tech_support",
    "online_security",
    "payment_method"
]

In [12]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [13]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            NUMERICAL_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        )
    ]
)

In [15]:
X_train_processed = preprocessor.fit_transform(X_train)

In [16]:
X_test_processed = preprocessor.transform(X_test)

In [17]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [18]:
logistic_model.fit(
    X_train_processed,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [19]:
logistic_predictions = logistic_model.predict(
    X_test_processed
)

In [20]:
logistic_probabilities = logistic_model.predict_proba(
    X_test_processed
)[:, 1]

In [21]:
logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

logistic_precision = precision_score(
    y_test,
    logistic_predictions
)

logistic_recall = recall_score(
    y_test,
    logistic_predictions
)

logistic_f1 = f1_score(
    y_test,
    logistic_predictions
)

logistic_roc_auc = roc_auc_score(
    y_test,
    logistic_probabilities
)

In [23]:
print("Logistic Regression")
print("-------------------")
print("Accuracy :", logistic_accuracy)
print("Precision:", logistic_precision)
print("Recall   :", logistic_recall)
print("F1       :", logistic_f1)
print("ROC-AUC  :", logistic_roc_auc)

Logistic Regression
-------------------
Accuracy : 0.731
Precision: 0.7497255762897914
Recall   : 0.9433701657458563
F1       : 0.8354740061162079
ROC-AUC  : 0.7120916406437666


In [24]:
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

In [25]:
random_forest_model.fit(
    X_train_processed,
    y_train
)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [26]:
rf_predictions = random_forest_model.predict(
    X_test_processed
)

In [27]:
rf_probabilities = random_forest_model.predict_proba(
    X_test_processed
)[:, 1]

In [28]:
rf_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

rf_precision = precision_score(
    y_test,
    rf_predictions
)

rf_recall = recall_score(
    y_test,
    rf_predictions
)

rf_f1 = f1_score(
    y_test,
    rf_predictions
)

rf_roc_auc = roc_auc_score(
    y_test,
    rf_probabilities
)

In [29]:
print("\nRandom Forest")
print("-------------")
print("Accuracy :", rf_accuracy)
print("Precision:", rf_precision)
print("Recall   :", rf_recall)
print("F1       :", rf_f1)
print("ROC-AUC  :", rf_roc_auc)


Random Forest
-------------
Accuracy : 0.711
Precision: 0.7508650519031141
Recall   : 0.899171270718232
F1       : 0.8183532369578881
ROC-AUC  : 0.6679628072703979


In [30]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    eval_metric="logloss"
)

In [32]:
xgb_model.fit(
    X_train_processed,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [33]:
xgb_predictions = xgb_model.predict(
    X_test_processed
)

In [34]:
xgb_probabilities = xgb_model.predict_proba(
    X_test_processed
)[:, 1]

In [35]:
xgb_accuracy = accuracy_score(
    y_test,
    xgb_predictions
)

xgb_precision = precision_score(
    y_test,
    xgb_predictions
)

xgb_recall = recall_score(
    y_test,
    xgb_predictions
)

xgb_f1 = f1_score(
    y_test,
    xgb_predictions
)

xgb_roc_auc = roc_auc_score(
    y_test,
    xgb_probabilities
)

In [36]:
print("\nXGBoost")
print("-------")
print("Accuracy :", xgb_accuracy)
print("Precision:", xgb_precision)
print("Recall   :", xgb_recall)
print("F1       :", xgb_f1)
print("ROC-AUC  :", xgb_roc_auc)


XGBoost
-------
Accuracy : 0.727
Precision: 0.7530864197530864
Recall   : 0.9267955801104972
F1       : 0.8309597523219814
ROC-AUC  : 0.6803637200736649


In [37]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        logistic_accuracy,
        rf_accuracy,
        xgb_accuracy
    ],
    "Precision": [
        logistic_precision,
        rf_precision,
        xgb_precision
    ],
    "Recall": [
        logistic_recall,
        rf_recall,
        xgb_recall
    ],
    "F1": [
        logistic_f1,
        rf_f1,
        xgb_f1
    ],
    "ROC-AUC": [
        logistic_roc_auc,
        rf_roc_auc,
        xgb_roc_auc
    ]
})

print(results)

                 Model  Accuracy  Precision    Recall        F1   ROC-AUC
0  Logistic Regression     0.731   0.749726  0.943370  0.835474  0.712092
1        Random Forest     0.711   0.750865  0.899171  0.818353  0.667963
2              XGBoost     0.727   0.753086  0.926796  0.830960  0.680364


In [38]:
best_model_name = results.loc[
    results["F1"].idxmax(),
    "Model"
]

print("Best model:", best_model_name)

Best model: Logistic Regression


In [39]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "XGBoost": xgb_model
}

best_model = models[best_model_name]

In [40]:
best_model

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [42]:
joblib.dump(
    best_model,
    "/Users/mitulshah/Downloads/mlops-churn-project/models/model.pkl"
)

['/Users/mitulshah/Downloads/mlops-churn-project/models/model.pkl']

In [43]:
loaded_model = joblib.load(
    "/Users/mitulshah/Downloads/mlops-churn-project/models/model.pkl"
)

print(type(loaded_model))

<class 'sklearn.linear_model._logistic.LogisticRegression'>
